# 特征值与特征向量计算实践

## 实验目标

通过编写代码深入理解并掌握：

- 特征对（特征值与特征向量）的定义验证方法。
- 矩阵迹（trace）与特征值之和、行列式与特征值之积的关系验证。
- 幂迭代法（Power Iteration）求主特征值的迭代过程。
- 矩阵对角化公式 $A = S\Lambda S^{-1}$ 的构造与验证。
- 利用对角化高效计算矩阵幂 $A^k$，并模拟马尔可夫链的稳态收敛。

## 限制条件

- **严禁**调用 `numpy.linalg.eig`、`numpy.linalg.eigvals` 等特征值分解函数来实现本作业中的核心算法（可作为验证标准答案使用）。
- 迹计算、马尔可夫模拟等核心操作必须从底层循环写起，不可直接使用 numpy 替代。
- 输入矩阵以 `list[list[float]]` 形式传入，禁止在核心函数内部使用 numpy 运算符。
- 任务 4 中允许调用 `numpy.linalg.eig` 获取特征值与特征向量（作为输入），允许调用 `numpy.linalg.inv` 计算 $S^{-1}$。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def print_matrix(name, matrix):
    print(f"---- {name} ----")
    for row in matrix:
        print([f"{val:8.4f}" for val in row])
    print()


def mat_multiply(A: list, B: list) -> list:
    """
    矩阵乘法 A * B
    参数：A - m×k 矩阵，B - k×n 矩阵
    返回：m×n 结果矩阵（list[list[float]]）

    温馨提示：这里提供了某位同学上次的作业代码示例，如需可自行更改
    """
    m = len(A)
    k = len(B)
    n = len(B[0])
    C = [[0.0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            for p in range(k):
                C[i][j] += A[i][p] * B[p][j]
    return C

---

## 任务 1：验证特征对定义 $Av = \lambda v$

**背景知识：** 若非零向量 $v$ 满足

$$Av = \lambda v$$

则称 $v$ 是矩阵 $A$ 的**特征向量**，$\lambda$ 是对应的**特征值**。

几何含义：特征向量在线性变换 $A$ 作用下方向不变（或反向），只是长度按因子 $|\lambda|$ 缩放。

**编程要求：**

- 补全 `matrix_vector_mult(A, v)`：计算矩阵与向量的乘积 $Av$。
- 补全 `check_eigenpair(A, lam, v, tol)`：验证 $(\lambda, v)$ 是否是矩阵 $A$ 的特征对，
  即检验残差 $\|Av - \lambda v\|$ 是否小于容差 `tol`。

In [ ]:
def matrix_vector_mult(A: list, v: list) -> list:
    """
    计算矩阵向量乘积 Av
    参数：A - m×n 矩阵（list[list[float]]），v - n 维向量（list[float]）
    返回：m 维结果向量（list[float]）
    """
    # TODO
    pass


def check_eigenpair(A: list, lam: float, v: list, tol: float = 1e-6) -> bool:
    """
    验证 (lam, v) 是否是矩阵 A 的特征对，即 Av ≈ λv
    参数：A   - n×n 矩阵（list[list[float]]）
         lam - 待验证的特征值（float）
         v   - 待验证的特征向量（list[float]）
         tol - 容差
    返回：布尔值，True 表示验证通过
    提示：计算残差向量 r = Av - λv，检验其每个分量绝对值均小于 tol
    """
    # TODO
    pass

In [ ]:
# 测试用例 1：投影矩阵 P
P = [[0.5, 0.5],
     [0.5, 0.5]]

v1 = [1.0,  1.0]   # 沿 y=x 方向，期望特征值 λ=1
v2 = [1.0, -1.0]   # 垂直方向，  期望特征值 λ=0

print("=== 投影矩阵 P ===")
print_matrix("P", P)

r1 = check_eigenpair(P, 1.0, v1)
r2 = check_eigenpair(P, 0.0, v2)
r3 = check_eigenpair(P, 2.0, v1)   # 错误特征值，期望 False

print(f"特征对 (λ=1,  v=[1, 1])  验证结果: {r1}，期望: True")
print(f"特征对 (λ=0,  v=[1,-1]) 验证结果: {r2}，期望: True")
print(f"错误对 (λ=2,  v=[1, 1])  验证结果: {r3}，期望: False")

print()

# 测试用例 2：反射矩阵 R
R = [[0.0, 1.0],
     [1.0, 0.0]]

print("=== 反射矩阵 R ===")
print_matrix("R", R)

r4 = check_eigenpair(R,  1.0, v1)
r5 = check_eigenpair(R, -1.0, v2)

print(f"特征对 (λ= 1, v=[1, 1])  验证结果: {r4}，期望: True")
print(f"特征对 (λ=-1, v=[1,-1]) 验证结果: {r5}，期望: True")

assert r1 and r2 and (not r3) and r4 and r5, "特征对验证存在错误！"
print("\n任务1 所有测试通过！")

---

## 任务 2：迹与行列式的特征值关系验证

**背景知识：** 对于 $n \times n$ 矩阵，特征值 $\lambda_1, \lambda_2, \ldots, \lambda_n$ 与矩阵有以下关系：

| 关系 | 公式 |
|:---:|:---:|
| 特征值之和 | $\text{trace}(A) = \sum_{i=1}^{n} A_{ii} = \lambda_1 + \lambda_2 + \cdots + \lambda_n$ |
| 特征值之积 | $\det(A) = \lambda_1 \cdot \lambda_2 \cdots \lambda_n$ |

**编程要求：**

- 补全 `compute_trace(A)`：计算矩阵的迹（对角线元素之和）。
- 使用 `numpy.linalg.eigvals`（此处允许用于获取参考特征值）及 `numpy.linalg.det` 验证上述两个公式。
- 浮点数比较请使用 `abs(a - b) < tol` 的方式，不得直接用 `==`。

In [ ]:
def compute_trace(A: list) -> float:
    """
    计算矩阵 A 的迹（对角线元素之和）
    参数：A - n×n 矩阵（list[list[float]]）
    返回：迹的值（float）
    """
    # TODO
    pass

In [ ]:
tol = 1e-6
np.random.seed(21)

print(f"{'矩阵':>6}  {'trace(A)':>12}  {'sum(λ)':>12}  {'迹匹配':>8}  {'det(A)':>14}  {'prod(λ)':>14}  {'行列式匹配':>10}")
print("-" * 90)

for n in [2, 3, 4, 5]:
    A_np = np.random.randint(-4, 5, (n, n)).astype(float)
    A = A_np.tolist()

    my_trace = compute_trace(A)
    eigvals  = np.linalg.eigvals(A_np)

    sum_lam  = sum(eigvals).real
    prod_lam = np.prod(eigvals).real
    det_A    = np.linalg.det(A_np)

    trace_ok = abs(my_trace - sum_lam) < tol
    det_ok   = abs(det_A - prod_lam) < tol * max(1.0, abs(det_A))

    print(f"{n}x{n}  {my_trace:>12.4f}  {sum_lam:>12.4f}  {str(trace_ok):>8}"
          f"  {det_A:>14.4f}  {prod_lam:>14.4f}  {str(det_ok):>10}")
    assert trace_ok, f"{n}x{n} 迹计算不正确！"

print("\n任务2 所有测试通过！")

---

## 任务 3：幂迭代法（Power Iteration）求主特征值

**背景知识：** 幂迭代法是求矩阵**绝对值最大特征值**（主特征值）的经典算法，步骤如下：

1. 初始化随机单位向量 $v^{(0)}$。
2. 重复迭代，直至收敛：
   $$v^{(k+1)} = \frac{A v^{(k)}}{\|A v^{(k)}\|}$$
3. 利用 **Rayleigh 商**估计特征值：
   $$\lambda^{(k)} \approx \frac{(v^{(k)})^\top A v^{(k)}}{(v^{(k)})^\top v^{(k)}}$$

当矩阵有唯一最大模特征值时，$v^{(k)}$ 收敛到对应的特征向量。

**编程要求：**

- 补全 `vec_norm(v)`：计算向量的 L2 范数 $\|v\| = \sqrt{\sum_i v_i^2}$。
- 补全 `power_iteration(A, num_iter, tol)`：返回 `(eigenvalue, eigenvector)`。
- 迭代终止条件：连续两次特征值估计之差的绝对值小于 `tol`，或达到最大迭代次数。
- **禁止**在核心迭代中使用任何 numpy 运算，向量归一化与乘法均需手动实现。

In [ ]:
def vec_norm(v: list) -> float:
    """
    计算向量 v 的 L2 范数
    参数：v - 向量（list[float]）
    返回：L2 范数（float）
    """
    # TODO
    pass


def power_iteration(A: list, num_iter: int = 1000, tol: float = 1e-10):
    """
    幂迭代法：求矩阵 A 的主特征值（绝对值最大的特征值）及其特征向量
    参数：A        - n×n 矩阵（list[list[float]]）
         num_iter - 最大迭代次数
         tol      - 收敛容差
    返回：(eigenvalue, eigenvector)
             eigenvalue  - 主特征值（float）
             eigenvector - 归一化特征向量（list[float]）
    提示：Rayleigh 商 = (v^T * Av) / (v^T * v) = sum(v[i] * Av[i]) / sum(v[i]^2)
    """
    import random
    n = len(A)
    random.seed(0)
    # 初始化随机向量并归一化
    v = [random.gauss(0, 1) for _ in range(n)]
    norm = vec_norm(v)
    v = [x / norm for x in v]

    lam_prev = 0.0  # 温馨提示：可能需要，也可自行更改

    for _ in range(num_iter):
        # 步骤1：计算 w = Av（使用 matrix_vector_mult）
        # TODO
        
        # 步骤2：计算 Rayleigh 商估计特征值
        #   lam = sum(v[i] * w[i]) / sum(v[i]^2)
        # TODO
        
        # 步骤3：归一化 w 得到新的 v
        # TODO
        
        # 步骤4：检查收敛条件
        # TODO
        
        pass

In [ ]:
np.random.seed(5)

print(f"{'矩阵':>6}  {'幂迭代特征值':>18}  {'numpy 参考值':>18}  {'误差':>12}")
print("-" * 65)

for n in [2, 3, 4, 5]:
    # 构造对称正定矩阵，保证最大模特征值唯一（实数）
    B = np.random.randint(-3, 4, (n, n)).astype(float)
    A_np = B @ B.T + np.eye(n) * n
    A = A_np.tolist()

    lam_my, vec_my = power_iteration(A)

    # numpy 参考：最大特征值
    lam_ref = max(np.linalg.eigvals(A_np).real)

    err = abs(lam_my - lam_ref)
    print(f"{n}x{n}  {lam_my:>18.6f}  {lam_ref:>18.6f}  {err:>12.2e}")
    assert err < 1e-4, f"{n}x{n} 幂迭代结果不正确！"

print("\n任务3 所有测试通过！")

---

## 任务 4：矩阵对角化与矩阵幂的高效计算

**背景知识：** 若矩阵 $A$ 有 $n$ 个线性无关的特征向量 $v_1, v_2, \ldots, v_n$，构造

$$S = [v_1 \;|\; v_2 \;|\; \cdots \;|\; v_n], \qquad
\Lambda = \begin{bmatrix} \lambda_1 & & \\ & \ddots & \\ & & \lambda_n \end{bmatrix}$$

则有**对角化分解**：

$$A = S\Lambda S^{-1}$$

利用这一分解，矩阵的 $k$ 次幂可以高效计算：

$$A^k = S\Lambda^k S^{-1}, \qquad \Lambda^k = \begin{bmatrix} \lambda_1^k & & \\ & \ddots & \\ & & \lambda_n^k \end{bmatrix}$$

**编程要求：**

- 补全 `build_diag_matrix(diag_vals)`：根据对角元素列表构造 $n \times n$ 对角矩阵。
- 补全 `mat_power_via_diag(A, k)`：利用对角化计算 $A^k$。
  - 允许调用 `numpy.linalg.eig` 获取特征值/特征向量，允许调用 `numpy.linalg.inv` 计算 $S^{-1}$。
  - 但所有矩阵乘法必须使用 `mat_multiply` 完成，对角矩阵幂次必须手动计算。

In [ ]:
def build_diag_matrix(diag_vals: list) -> list:
    """
    根据对角元素列表构造 n×n 对角矩阵
    参数：diag_vals - 对角元素列表（list[float]），长度为 n
    返回：n×n 对角矩阵（list[list[float]]），非对角元素为 0
    """
    # TODO
    pass


def mat_power_via_diag(A: list, k: int) -> list:
    """
    利用对角化 A = S Λ S^{-1} 计算 A^k = S Λ^k S^{-1}
    参数：A - n×n 可对角化矩阵（list[list[float]]），k - 非负整数
    返回：A^k（list[list[float]]）
    步骤提示：
      1. 用 numpy.linalg.eig 得到特征值数组和特征向量矩阵（S）
      2. 取实部后转换为 list[list[float]]
      3. 用 build_diag_matrix 构造 Λ，各元素取 k 次幂得到 Λ^k
      4. 用 numpy.linalg.inv 计算 S^{-1}，转为 list[list[float]]
      5. 用 mat_multiply 计算 S * Λ^k * S^{-1}
    """
    # TODO
    pass

In [ ]:
# 测试用例1：对角化公式验证 A = S Λ S^{-1}
print("=== 对角化重构验证 A = SΛS⁻¹ ===")
A_test = [[4.0, 1.0],
          [2.0, 3.0]]

eigvals_np, eigvecs_np = np.linalg.eig(np.array(A_test))
S_list     = eigvecs_np.real.tolist()
Lambda_k   = build_diag_matrix(eigvals_np.real.tolist())   # Λ^1 = Λ
S_inv_list = np.linalg.inv(eigvecs_np).real.tolist()

A_recon = mat_multiply(mat_multiply(S_list, Lambda_k), S_inv_list)

print("原矩阵 A:")
print_matrix("A", A_test)
print("重构矩阵 SΛS⁻¹:")
print_matrix("SΛS⁻¹", A_recon)

err_recon = max(abs(A_recon[i][j] - A_test[i][j]) for i in range(2) for j in range(2))
print(f"重构误差（最大元素误差）: {err_recon:.2e}")
assert err_recon < 1e-8, "对角化重构误差过大！"

print()

# 测试用例2：矩阵幂 A^k 与 numpy 参考值对比
print("=== 矩阵幂 A^k 验证 ===")
A_pow    = [[2.0, 1.0], [1.0, 3.0]]
A_pow_np = np.array(A_pow)

print(f"{'k':>5}  {'最大元素误差':>16}")
print("-" * 25)
for k in [1, 2, 5, 10, 20]:
    Ak_my  = mat_power_via_diag(A_pow, k)
    Ak_ref = np.linalg.matrix_power(A_pow_np, k).tolist()
    err = max(abs(Ak_my[i][j] - Ak_ref[i][j]) for i in range(2) for j in range(2))
    print(f"{k:>5}  {err:>16.2e}")
    assert err < 1e-4, f"A^{k} 计算误差过大！"

print("\n任务4 所有测试通过！")

---

## 任务 5：马尔可夫链稳态模拟

**背景知识：** 马尔可夫链描述了系统状态随时间演化的过程：

$$u_{k+1} = M u_k$$

其中 $M$ 是**转移矩阵**（每列之和为1，所有元素非负），$u_k$ 是第 $k$ 步的状态向量（各分量之和为1）。

马尔可夫矩阵一定有特征值 $\lambda_1 = 1$（最大特征值），经过足够多次迭代后，系统收敛到**稳态**：

$$u_\infty = \lim_{k \to \infty} M^k u_0 \propto v_1$$

其中 $v_1$ 是对应 $\lambda = 1$ 的特征向量（归一化使分量之和为1）。

**示例**（城市-郊区人口迁移模型）：

$$M = \begin{bmatrix} 0.8 & 0.3 \\ 0.2 & 0.7 \end{bmatrix}$$

特征值 $\lambda_1 = 1$，$\lambda_2 = 0.5$；稳态特征向量 $v_1 = [3, 2]^\top$，归一化后为 $[0.6, 0.4]^\top$。

**编程要求：**

- 补全 `simulate_markov(M, u0, steps)`：模拟马尔可夫链，返回每步的状态向量历史（共 `steps+1` 个向量，含初始状态）。
- 运行仿真并绘图，观察城市与郊区人口比例的稳态收敛过程。

In [ ]:
def simulate_markov(M: list, u0: list, steps: int) -> list:
    """
    模拟马尔可夫链：u_{k+1} = M * u_k，共 steps 步
    参数：M     - n×n 转移矩阵（list[list[float]]）
         u0    - 初始状态向量（list[float]），分量之和应为 1
         steps - 迭代步数
    返回：状态历史列表（list[list[float]]），
          第 0 个元素为初始状态 u0，第 k 个元素为第 k 步状态 u_k
          共 steps+1 个向量
    提示：每步用 matrix_vector_mult 计算下一步状态，并追加到 history
    """

    # TODO
    pass

In [ ]:
# 城市-郊区人口迁移模型（讲义示例）
M  = [[0.8, 0.3],
      [0.2, 0.7]]
u0 = [0.9, 0.1]   # 初始：城市 90%，郊区 10%

history = simulate_markov(M, u0, 50)

city   = [u[0] for u in history]
suburb = [u[1] for u in history]

# 理论稳态：v1=[3,2] 归一化后 [0.6, 0.4]
steady_city   = 3 / 5
steady_suburb = 2 / 5

print(f"模拟稳态（第50步）：城市 = {city[-1]:.6f}，郊区 = {suburb[-1]:.6f}")
print(f"理论稳态：          城市 = {steady_city:.6f}，郊区 = {steady_suburb:.6f}")

assert abs(city[-1]   - steady_city)   < 1e-4, "城市稳态不正确！"
assert abs(suburb[-1] - steady_suburb) < 1e-4, "郊区稳态不正确！"

# ---- 绘图 ----
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # 支持中文显示
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(8, 4))
plt.plot(city,   'o-', markersize=3, label="城市人口比例")
plt.plot(suburb, 's-', markersize=3, label="郊区人口比例")
plt.axhline(steady_city,   color='steelblue', linestyle='--', alpha=0.6,
            label=f"城市稳态 = {steady_city}")
plt.axhline(steady_suburb, color='darkorange', linestyle='--', alpha=0.6,
            label=f"郊区稳态 = {steady_suburb}")
plt.xlabel("迭代步数")
plt.ylabel("人口比例")
plt.title("马尔可夫链：城市-郊区人口迁移稳态收敛")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n任务5 所有测试通过！")